# CASM end-to-end verification notebook

Walks the full pipeline against a baseline observation. Designed for the user to inspect before any merge of the `refactor-tower` branch.

Stages (in order): CAsMan refresh → load + diagnostics → fringe-stop → SVD calibrate → BF weights + inspect + transit + deploy (DADA only) → imaging → layout-version trace.

Live deploy to `corr1`/`corr2` is **explicitly commented out**. Uncomment only when you intend to push weights live.

Run from the integration venv:

```bash
source /home/casm/software/dev/casm_venvs/casm_refactor_env/bin/activate
jupyter lab notebooks/end_to_end.ipynb
```

## 1. Setup

In [ ]:
%matplotlib inline

import os
from pathlib import Path

# Proxy env vars are required for run_sync_wiring's GitHub fetch.
os.environ.setdefault("HTTP_PROXY",  "http://10.70.0.10:8118")
os.environ.setdefault("HTTPS_PROXY", "http://10.70.0.10:8118")

# Optional: pin the canonical layout if you want a specific one.
# os.environ["CASM_LAYOUT_CSV"] = "/home/casm/software/dev/antenna_layouts/casm_antenna_layout_may2026.csv"

# Imports across all five repos.
from casm_io.correlator import read_visibilities, AntennaMapping, load_format
from casm_io.constants import OVRO, C_LIGHT_M_PER_NS, resolve_layout_path

from casm_vis_analysis import (
    run_sync_wiring, run_build_layout,
    plot_autocorr, plot_waterfall, plot_fringe_diag, plot_phase_vs_freq,
    fringe_stop, FringeStoppedData,
    coherence_metric, auto_detect_sign,
    fit_delay, apply_delay,
    source_flux, RFIMask,
)
from casm_calibrator import (
    SVDCalibrator, SVDConfig, CalibrationResult, CalibrationWeightsWriter,
)
from bf_weights_generator import BFWeights, inspect_snap_weights
from casm_imaging import make_phased_sum_image

import matplotlib.pyplot as _plt
print('matplotlib backend:', _plt.get_backend())
print('Layout will resolve to:', resolve_layout_path())
print('OVRO site:', OVRO)
print('C / 1 ns =', C_LIGHT_M_PER_NS, 'm')

## 2. CAsMan refresh (dry-run by default)

In [ ]:
# Pull a fresh CAsMan DB and show the wiring diff.
diff = run_sync_wiring(dry_run=True, force_pull=True)
print('added:',   len(diff['added']))
print('removed:', len(diff['removed']))
print('changed:', len(diff['changed']))

In [ ]:
# Build the consumer layout CSV (writes a dated file + updates the
# `current` symlink in $CASM_LAYOUT_DIR).
out = run_build_layout(dated=True, update_symlink=True, check_casman=True)
print(f"output: {out['output_csv']}  ({out['n_total']} rows, {out['n_active']} active)")
print(f"symlink updated: {out['symlink_updated']}")

## 3. Load + diagnostics

In [ ]:
# Pick the baseline observation here.
TIME_START = '2026-05-06 06:00:00'
TIME_END   = '2026-05-06 10:00:00'
TIME_TZ    = 'America/Los_Angeles'
DATA_ROOT  = '/mnt'
FORMAT     = '/home/casm/software/dev/casm_io/casm_io/correlator/configs/layout_64ant.json'

fmt = load_format(FORMAT)
ant = AntennaMapping.load()      # canonical resolver: $CASM_LAYOUT_CSV / current symlink
data = read_visibilities(
    time_start=TIME_START, time_end=TIME_END, time_tz=TIME_TZ,
    data_root=DATA_ROOT, fmt=fmt,
    freq_range_mhz=(60.0, 80.0),    # real on-disk channel skip via memmap
)
print(f'vis:       {data.vis.shape}')
print(f'freq_mhz:  {data.freq_mhz.shape}  [{data.freq_mhz[0]:.1f} -> {data.freq_mhz[-1]:.1f}]')
print(f'time_unix: {data.time_unix.shape}')
print(f'antenna mapping: {ant}')

In [ ]:
# Compose API: one read, many plots.
auto_figs = plot_autocorr(data, ant, scale='dB', ncols=3)
wf_figs   = plot_waterfall(data, ant, split_max=22)

## 4. Fringe-stop

In [ ]:
# For the fringe-stop wrapper, data must be ref+target-filtered.
# The simplest way is to re-read with explicit ref/targets, OR use
# the runner mirror which handles it for you. Demonstrating the runner
# here so the cell runs end-to-end without manual baseline indexing.
from casm_vis_analysis import run_fringe_stop
fs = run_fringe_stop(
    format=FORMAT, layout=str(out['output_csv']),
    ref_ant=10, source='sun', sign=-1,
    time_start=TIME_START, time_end=TIME_END, time_tz=TIME_TZ,
    data_root=DATA_ROOT,
    freq_range_mhz=(60.0, 80.0),
    delay_model=['linear'], antenna_delays=True,
    show=True,
)
print('fs keys:', list(fs.keys()))

## 5. SVD calibrate

The compose-style `svd_calibrate(fs, ant)` is staged but the full dict-wrapper lands with the VisibilityMatrix dedup. For the verification run, use `SVDCalibrator` directly with a `VisibilityMatrix` you build yourself (legacy path).

In [ ]:
# Stub: legacy SVD path (replace with svd_calibrate(fs, ant) when the
# matrix-construction wrapper lands on this branch).
# config = SVDConfig(threshold=4.0, ref_ant_idx=0)
# calibrator = SVDCalibrator(config)
# vis_matrix = ... # build (n_chan, n_ant, n_ant) Hermitian from fs
# svd_result = calibrator.calibrate(vis_matrix.vis_avg)
# CalibrationWeightsWriter().write(
#     'cal_demo.npz', svd_result, freqs_mhz=fs['freq_mhz'],
#     ant_ids=..., ref_ant_id=10, source='sun',
# )
print('SVD step is staged; uncomment once the dict-wrapper lands.')

## 6. Beamforming weights (full recipe up to deploy)

Builds on `cal` from the SVD section above. Steps:

1. Save cal as HDF5 (the format `bf_weights_generator` consumes).
2. Pick the beam grid (3 idioms shown).
3. Combine geometric + cal weights.
4. Save complex64 + int8 HDF5 files.
5. Inspect what landed.
6. Plot source transits across the beams.
7. Stage DADA files for deploy (live-upload cell stays commented out).

In [ ]:
# 6.1  Save the SVD calibration as HDF5 (the form bf_weights_generator wants).
from casm_calibrator import save_calibration

CAL_H5 = '/tmp/cal_demo.h5'
save_calibration(cal, CAL_H5, n_time_averaged=fs['vis_stopped'].shape[0])
print('wrote', CAL_H5)

In [ ]:
# 6.2  Pick a beam grid. Three idioms — uncomment the one you want.
from bf_weights_generator import (
    StationaryPointing,
    generate_beam_grid_altaz,    # rectangular alt/az grid
    generate_beam_grid_lm,       # uniform direction-cosine grid
    generate_beam_grid,          # target n_beams + array-aware FWHM
    Array64Config,
)

# (a) Explicit pointings (e.g. transit at known sources):
# beams = [
#     StationaryPointing.from_altaz(60.0, 180.0, name='sun_60S'),
#     StationaryPointing.from_altaz(45.0,  90.0, name='ne_45'),
# ]

# (b) Uniform alt/az grid:
# beams = generate_beam_grid_altaz(
#     alt_min_deg=30.0, alt_max_deg=85.0,
#     az_min_deg=0.0,   az_max_deg=360.0,
#     spacing_deg=10.0, exclude_horizon=True,
# )

# (c) Direction-cosine grid (uniform sky coverage):
# beams = generate_beam_grid_lm(spacing=0.07)

# (d) Target a beam count + auto-spacing from the array geometry:
arr = Array64Config.from_csv(str(out['output_csv']))
beams = generate_beam_grid(
    n_beams=64,
    alt_min_deg=30.0, alt_max_deg=90.0,
    array_config=arr,
    freq_hz=437.5e6,
)
print(f'{len(beams)} beams')

In [ ]:
# 6.3  Combine geometry + cal -> complex64 weights ready for the F-engine.
from bf_weights_generator import (
    load_calibration_weights, generate_combined_weights,
    save_combined_weights_hdf5, save_int8_weights_hdf5,
)

cal_weights = load_calibration_weights(CAL_H5)

combined = generate_combined_weights(
    pointing=beams,
    array_config=arr,
    cal_weights=cal_weights,
    freq_order='descending',
)
print('combined.weights shape:', combined.weights.shape,
      '  (n_beams, 64 SNAP slots, n_chan)')

In [ ]:
# 6.4  Save: complex64 (full precision) + int8 (deployment quantization).
WEIGHTS_H5      = '/tmp/my_weights.h5'
WEIGHTS_INT8_H5 = '/tmp/my_weights_int8.h5'
save_combined_weights_hdf5(combined, WEIGHTS_H5,      overwrite=True)
save_int8_weights_hdf5    (combined, WEIGHTS_INT8_H5, overwrite=True)
print('wrote', WEIGHTS_H5)
print('wrote', WEIGHTS_INT8_H5)

In [ ]:
# 6.5  Sanity-check the int8 file: alt/az coverage, beam count,
#      which SNAP inputs carry non-zero weights.
from bf_weights_generator import inspect_snap_weights

info = inspect_snap_weights(WEIGHTS_INT8_H5,
                            layout=str(out['output_csv']))
print()
print(f"#beams: {info['n_beams']}")
print(f"alt range: {min(info['beam_alt_deg']):.1f}° – "
      f"{max(info['beam_alt_deg']):.1f}°")
print(f"az  range: {min(info['beam_az_deg']):.1f}° – "
      f"{max(info['beam_az_deg']):.1f}°")
print(f"non-zero SNAP inputs: {info['n_nonzero_snap_inputs']} / 64")

In [ ]:
# 6.6  Where do bright sources transit through these beams today?
from bf_weights_generator import plot_source_transit

fig = plot_source_transit(
    WEIGHTS_INT8_H5,
    sources=['sun', 'cas-a', 'cyg-a'],
    date='2026-05-07',                   # YYYY-MM-DD; or omit for today
    time_tz='America/Los_Angeles',
    time_start='06:00', time_end='15:00',
    output_path=None,                    # render inline
)

In [ ]:
# 6.7  Stage DADA files (the format the SNAP F-engine consumes).
#      `deploy_bf_weights` is uncommitted on production main, so for
#      now invoke the script form. It writes direct.dada.{0..5} into
#      the cwd (or --output-dir).
import subprocess, sys, os
DEPLOY_OUT = '/tmp/snap_outputs'
os.makedirs(DEPLOY_OUT, exist_ok=True)
subprocess.run([
    sys.executable,
    '/home/casm/software/dev/bf_weights_generator/deploy_bf_weights.py',
    WEIGHTS_INT8_H5,
    '--output-dir', DEPLOY_OUT,
    '--dry-run',
], check=False)
print('staged DADA files in', DEPLOY_OUT)

In [ ]:
# ----- LIVE DEPLOY: DO NOT RUN unless you intend to push to corr1/corr2 -----
#
# subprocess.run([
#     sys.executable,
#     '/home/casm/software/dev/bf_weights_generator/deploy_bf_weights.py',
#     WEIGHTS_INT8_H5,
#     '--upload',
#     '--save-defaults',
#     '--utc-start', '2026-MM-DD-HH:MM:SS',
# ], check=True)

## 7. Imaging

In [ ]:
# CLI mirror: `casm-bf-image --source sun --start ... --end ...`
img = make_phased_sum_image(
    source='sun',
    time_start=TIME_START,
    time_end=TIME_END,
    output_dir='./images',
    ang_max_deg=15.0,
    npix=101,
    fs_sign=-1,
    fast_mode=True,
    compute_snr=True,
    verbose=True,
)
if img.get('snr_info') is not None:
    print('Measured SNR:', img['snr_info']['snr'])

## 8. Layout-version trace

In [ ]:
# Confirm every artifact references the same canonical CSV.
print('canonical layout (resolved):', resolve_layout_path())
print('build_layout output_csv:    ', out['output_csv'])
# When cal NPZ + weights HDF5 are produced, also print their layout_version stamps:
# import numpy as np
# cal = np.load('cal_demo.npz', allow_pickle=True)
# print('cal layout_version:', dict(cal['layout_version'].item()))